## tl;dr
独立求职研究作品，研究快照 2026-10-06。JEPX 643 个完整日、30,864 半小时时段，系统价格及五区域；默认中部 2025 年理论套利 17.76 百万 JPY。所有收益均为已知历史价格的研究回测，不是项目利润。

## Context & Methods
面向上海淘科新能源的研究及量化岗位。此伴读复核官方原始数据、研究区间和模型输出。全套年度模型在 `scripts/analyze.mjs` 生成，本伴读独立重算一个真实日。Python 3 标准库与现代 Node.js 即可运行；没有凭证或网络依赖。

### Key Assumptions
2MW/8.14MWh/中部是公司公开案例规格；效率88%、SOC10%-90%、每日首尾10%、循环上限1次、变动成本0均为 Demo 假设。每日独立优化、80格SOC网格、首尾能量守恒，固定运维、网络费、市场费、税和融资未计。

In [1]:
from pathlib import Path
import csv, gzip, hashlib, json, subprocess
from collections import Counter
from datetime import date, timedelta
ROOT = Path.cwd()
if not (ROOT / 'dist/data/quality.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'dist/data/quality.json').exists(), '从模块目录或research目录运行'
quality = json.loads((ROOT / 'dist/data/quality.json').read_text(encoding='utf-8'))
summary = json.loads((ROOT / 'dist/data/summary.json').read_text(encoding='utf-8'))
print('区间:', quality['delivery_start'], '至', quality['delivery_end'])
print('单位:', quality['price_unit'], '| 时区:', quality['timezone'])

区间: 2025-01-01 至 2026-10-05
单位: JPY/kWh | 时区: Asia/Tokyo


## Data
来源：JEPX 官方现货页面 https://www.jepx.jp/electricpower/market-data/spot/ 。官方图表 CSV 端点和抓取时间保存在质量 JSON。FY 文件从4月开始，规范化后筛选自然日；原始未来受渡记录保留但不进入研究区间。

In [2]:
for source in quality['sources']:
    raw = gzip.decompress((ROOT / f"research/raw/spot_summary_{source['fiscal_year']}.csv.gz").read_bytes())
    assert hashlib.sha256(raw).hexdigest() == source['raw_sha256']
    print('FY', source['fiscal_year'], '原始SHA-256一致；行数', source['source_row_count'])
with (ROOT / 'research/jepx-normalized.csv').open(encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))
keys = [(r['delivery_date_jst'], int(r['slot_1_to_48'])) for r in rows]
assert len(set(keys)) == len(keys) == 30864
day_slots = {}
for d,s in keys:
    day_slots.setdefault(d,set()).add(s)
assert all(s == set(range(1,49)) for s in day_slots.values())
expected = {(date(2025,1,1)+timedelta(days=i)).isoformat() for i in range(643)}
assert set(day_slots) == expected
assert all(all(v != '' for v in r.values()) for r in rows)
print('独立检查:',len(day_slots),'日；',len(rows),'时段；零缺失/重复；每天48点')

FY 2024 原始SHA-256一致；行数 17520
FY 2025 原始SHA-256一致；行数 17520
FY 2026 原始SHA-256一致；行数 9120
独立检查: 643 日； 30864 时段；零缺失/重复；每天48点


## Results
以下年度值读取同一确定性算法的冻结结果，按日均收益比较全年与 YTD。不是把YTD总额年化，也不是未来预测。

In [3]:
for window in ['2025','2026ytd']:
    print('区间',window,summary['windows'][window]['start'],summary['windows'][window]['end'])
    for region,r in summary['windows'][window]['regions'].items():
        assert abs(r['netRevenue'] - sum(d['netRevenue'] for d in r['daily'])) < 1e-5
        print(f"  {region:7s} {r['analyzedDays']:3d}天 | {r['netRevenue']/1e6:6.2f} 百万JPY | {r['netRevenue']/r['analyzedDays']:9,.0f} JPY/日")

区间 2025 2025-01-01 2025-12-31
  chubu   365天 |  17.76 百万JPY |    48,667 JPY/日
  tokyo   365天 |  14.36 百万JPY |    39,337 JPY/日
  kansai  365天 |  19.77 百万JPY |    54,151 JPY/日
  kyushu  365天 |  22.36 百万JPY |    61,273 JPY/日
  tohoku  365天 |  19.48 百万JPY |    53,376 JPY/日
区间 2026ytd 2026-01-01 2026-10-05
  chubu   278天 |  21.01 百万JPY |    75,579 JPY/日
  tokyo   278天 |  21.30 百万JPY |    76,625 JPY/日
  kansai  278天 |  19.69 百万JPY |    70,822 JPY/日
  kyushu  278天 |  22.72 百万JPY |    81,734 JPY/日
  tohoku  278天 |  26.32 百万JPY |    94,676 JPY/日


### 独立重算单日并积分物理约束
2025-07-15中部，原始48点；调用同一公开模型代码，以Python独立积分核对功率、效率、SOC、周期和现金流。

In [4]:
script = """import {readFileSync} from 'node:fs';
import {solveDay,DEFAULTS} from './dist/model.js';
const d=JSON.parse(readFileSync('./dist/data/prices.json')).days.find(d=>d.date==='2025-07-15');
console.log(JSON.stringify({parameters:DEFAULTS,...solveDay(d.chubu,DEFAULTS)}));"""
result = json.loads(subprocess.run(['node','--input-type=module','-e',script],cwd=ROOT,check=True,capture_output=True,text=True).stdout)
p = result['parameters']; eta=p['efficiency']**0.5; energy=p['energy']*p['initialSoc']; discharge_battery=0
for row in result['rows']:
    assert min(row['chargeMW'],row['dischargeMW']) < 1e-9
    assert max(row['chargeMW'],row['dischargeMW']) <= p['power']+1e-9
    assert abs(energy/p['energy']-row['socBefore']) < 1e-9
    energy += row['chargeMW']*0.5*eta-row['dischargeMW']*0.5/eta
    discharge_battery += row['dischargeMW']*0.5/eta
    assert p['socMin']-1e-9 <= energy/p['energy'] <= p['socMax']+1e-9
    assert abs(energy/p['energy']-row['socAfter']) < 1e-9
assert abs(energy-p['energy']*p['initialSoc']) < 1e-9
assert discharge_battery <= p['energy']*p['maxCycles']+1e-9
assert abs(sum(x['cashflow']-x['variableCost'] for x in result['rows'])-result['netRevenue']) < 1e-5
print('48时段物理积分与现金流通过；毛套利',round(result['grossRevenue']),'JPY')
print('首尾SOC',result['rows'][0]['socBefore'],result['rows'][-1]['socAfter'])
validation=json.loads((ROOT / 'dist/data/model-validation.json').read_text())
assert validation['status']=='passed' and len(validation['cases'])==7
print('7日独立连续LP对照；最大网格差异',round(validation['max_relative_grid_loss']*100,3),'%（抽样，不是普遍误差保证）')

48时段物理积分与现金流通过；毛套利 114272 JPY
首尾SOC 0.1 0.1
7日独立连续LP对照；最大网格差异 2.978 %（抽样，不是普遍误差保证）


## Takeaways
中部默认2025基线17.76百万JPY；同约束2025九州最高，2026YTD东北最高。区域排名变化，单年排名不足以决定选址。工程和成本假设必须单独核验。

EPRX规则、官方成交统计及项目阶段证据见 `dist/data/evidence.json` 和8页Memo；调频仅做容量情景，不能把现货和调频收入相加。不得将模型收益归因于淘科实际经营。